In [1]:
# ==========================================================================
# California Housing 가격 예측 : 학습 -> 저장 -> 모듈화 -> FastAPI 서빙 -> Streamlit 화면 -> 통합 테스트
#   서버 : app/housing_api.py -> http://localhost:8000  /  화면 : frontend/app_housing.py -> http://localhost:8501
# ==========================================================================
# [1. 패키지] scikit-learn : 데이터셋(fetch_california_housing)과 학습/테스트 분할에 씁니다
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
# [1. 데이터 둘러보기] 샘플 20,640개, 피처 8개, 타겟은 중위 주택 가격($100,000 단위)
#   첫 실행 때 데이터를 내려받아 ~/scikit_learn_data 에 캐시합니다
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()
print(f"샘플 수: {data.data.shape[0]:,}")
print(f"피처 수: {data.data.shape[1]}")
print(f"타겟: {data.target_names}")
print(f"\n피처 목록:")
for i, name in enumerate(data.feature_names):
    print(f"  {i+1}. {name}")

샘플 수: 20,640
피처 수: 8
타겟: ['MedHouseVal']

피처 목록:
  1. MedInc
  2. HouseAge
  3. AveRooms
  4. AveBedrms
  5. Population
  6. AveOccup
  7. Latitude
  8. Longitude


In [3]:
# [2. 데이터 준비] X(피처 행렬), y(타겟)를 꺼내고 피처별 평균/표준편차를 봅니다
#   피처마다 단위가 크게 달라(Population 은 수천, AveBedrms 는 1 내외) 정규화가 필요합니다
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

# 데이터 로드
data = fetch_california_housing()
X, y = data.data, data.target
feature_names = data.feature_names

print(f"피처 크기: {X.shape}")        # (20640, 8)
print(f"타겟 크기: {y.shape}")        # (20640,)
print(f"타겟 범위: {y.min():.2f} ~ {y.max():.2f} ($100,000 단위)")
print(f"\n피처별 통계:")
for i, name in enumerate(feature_names):
    print(f"  {name:12s}  평균: {X[:, i].mean():10.2f}  표준편차: {X[:, i].std():10.2f}")

피처 크기: (20640, 8)
타겟 크기: (20640,)
타겟 범위: 0.15 ~ 5.00 ($100,000 단위)

피처별 통계:
  MedInc        평균:       3.87  표준편차:       1.90
  HouseAge      평균:      28.64  표준편차:      12.59
  AveRooms      평균:       5.43  표준편차:       2.47
  AveBedrms     평균:       1.10  표준편차:       0.47
  Population    평균:    1425.48  표준편차:    1132.43
  AveOccup      평균:       3.07  표준편차:      10.39
  Latitude      평균:      35.63  표준편차:       2.14
  Longitude     평균:    -119.57  표준편차:       2.00


In [4]:
# [2.1 학습/테스트 분할] 80% 학습, 20% 테스트
#   random_state 를 고정해야 매번 같은 행이 테스트로 가서 결과를 비교할 수 있습니다
# 학습/테스트 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42  # *your code* — test_size와 random_state 설정
)

print(f"학습 데이터: {X_train.shape[0]:,}개")
print(f"테스트 데이터: {X_test.shape[0]:,}개")

학습 데이터: 16,512개
테스트 데이터: 4,128개


In [5]:
# [2.1 참고] random_state 를 바꾸면 테스트에 들어가는 행이 달라지는 것을 원본 인덱스로 확인합니다
#   주의 : 이 셀이 X_train/X_test 를 마지막 seed(123) 기준으로 덮어씁니다
#          이후 셀은 seed=123 분할로 학습/평가됩니다
import numpy as np
from sklearn.model_selection import train_test_split

# 원본 데이터의 행 번호 생성
indices = np.arange(len(X))

for seed in [42, 100, 123]:
    X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
        X, y, indices,
        test_size=0.2,
        random_state=seed
    )

    print(f"\nrandom_state={seed}")
    print(f"학습 데이터: {len(X_train):,}개")
    print(f"테스트 데이터: {len(X_test):,}개")
    print(f"테스트 데이터 원본 인덱스: {test_idx[:10].tolist()}")


random_state=42
학습 데이터: 16,512개
테스트 데이터: 4,128개
테스트 데이터 원본 인덱스: [20046, 3024, 15663, 20484, 9814, 13311, 7113, 7668, 18246, 5723]

random_state=100
학습 데이터: 16,512개
테스트 데이터: 4,128개
테스트 데이터 원본 인덱스: [8151, 53, 3039, 9484, 9307, 7471, 6823, 13359, 19408, 20249]

random_state=123
학습 데이터: 16,512개
테스트 데이터: 4,128개
테스트 데이터 원본 인덱스: [19121, 20019, 15104, 3720, 8938, 13730, 7955, 15109, 14354, 4450]


In [6]:
# [2.2 정규화 통계] 평균/표준편차는 반드시 '학습 데이터'에서만 구합니다
#   테스트 데이터의 통계를 쓰면 평가에 미래 정보가 섞입니다 (데이터 누수)
# 학습 데이터의 평균/표준편차 계산 (테스트 데이터에는 학습 데이터의 통계를 사용)
train_mean = X_train.mean(axis=0)  # *your code* — 축(axis) 설정
train_std = X_train.std(axis=0)    # *your code* — 축(axis) 설정

print("피처별 평균:", np.round(train_mean, 2))
print("피처별 표준편차:", np.round(train_std, 2))

피처별 평균: [ 3.87000e+00  2.85700e+01  5.42000e+00  1.10000e+00  1.43412e+03
  3.06000e+00  3.56300e+01 -1.19570e+02]
피처별 표준편차: [1.90000e+00 1.25800e+01 2.52000e+00 4.90000e-01 1.13028e+03 1.06200e+01
 2.13000e+00 2.00000e+00]


In [7]:
# [2.2 정규화 적용] (x - 평균) / 표준편차 -> 피처마다 평균 0, 표준편차 1 근처로 맞춥니다
# 정규화 적용
X_train_norm = (X_train - train_mean) / train_std  # *your code* — 정규화 공식
X_test_norm = (X_test - train_mean) / train_std     # 테스트에도 학습 데이터의 통계 사용

print(f"정규화 후 학습 데이터 평균: {X_train_norm.mean(axis=0).round(4)}")  # 거의 0
print(f"정규화 후 학습 데이터 표준편차: {X_train_norm.std(axis=0).round(4)}")  # 거의 1

정규화 후 학습 데이터 평균: [-0. -0.  0.  0.  0.  0. -0. -0.]
정규화 후 학습 데이터 표준편차: [1. 1. 1. 1. 1. 1. 1. 1.]


In [8]:
# [2.3 텐서 변환] numpy -> torch.FloatTensor
#   타겟은 모델 출력 (N, 1) 과 모양을 맞추려고 unsqueeze(1) 로 열 차원을 하나 더합니다
# 텐서 변환
X_train_tensor = torch.FloatTensor(X_train_norm)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)  # *your code* — (N,) → (N,1)
X_test_tensor = torch.FloatTensor(X_test_norm)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

print(f"X_train 텐서: {X_train_tensor.shape}")  # torch.Size([16512, 8])
print(f"y_train 텐서: {y_train_tensor.shape}")  # torch.Size([16512, 1])

X_train 텐서: torch.Size([16512, 8])
y_train 텐서: torch.Size([16512, 1])


In [9]:
# [3. 모델 정의] 8 -> 64 -> 32 -> 1 의 MLP 회귀 모델
#   ReLU : 비선형성 / Dropout(0.2) : 과적합 방지 / 마지막 층은 활성화 없이 실수 하나를 출력
class HousingModel(nn.Module):
    """캘리포니아 주택 가격 예측 모델 (회귀)"""
    def __init__(self, input_dim=8):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),   # *your code* — 입력 차원 → 64
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),           # *your code* — 32 → 1 (회귀 출력)
        )

    def forward(self, x):
        return self.network(x)

model = HousingModel(input_dim=8)
print(f"모델 구조:\n{model}")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

모델 구조:
HousingModel(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)
파라미터 수: 2,689


In [10]:
# [3.1 학습 설정] DataLoader 로 256개씩 섞어서 배치 학습
#   회귀 문제라 손실은 MSE(평균 제곱 오차), 옵티마이저는 Adam(lr=0.001)
# 학습 설정
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

criterion = nn.MSELoss()                                          # *your code* — 회귀이므로 MSELoss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)         # *your code* — Adam 옵티마이저

EPOCHS = 50

In [11]:
# [3.2 학습 루프] 배치마다 : 기울기 초기화 -> 순전파 -> 손실 -> 역전파 -> 가중치 갱신
#   10 에폭마다 평균 손실을 출력합니다
# 학습 루프
model.train()
for epoch in range(1, EPOCHS + 1):
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)          # *your code* — 순전파
        loss = criterion(predictions, y_batch) # *your code* — 손실 계산
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    if epoch % 10 == 0:
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch:3d}/{EPOCHS} — Loss: {avg_loss:.4f}")

Epoch  10/50 — Loss: 0.5239


Epoch  20/50 — Loss: 0.4589


Epoch  30/50 — Loss: 0.4204


Epoch  40/50 — Loss: 0.3971


Epoch  50/50 — Loss: 0.3795


In [12]:
# [3.3 테스트 평가] eval() 로 Dropout 을 끄고, no_grad() 로 기울기 계산 없이 예측합니다
#   MAE : 예측이 실제 가격에서 평균적으로 얼마나 벗어나는지 ($100,000 단위 -> 달러로 환산)
# 테스트 평가
model.eval()
with torch.no_grad():
    test_preds = model(X_test_tensor)
    test_loss = criterion(test_preds, y_test_tensor)

    # MAE (Mean Absolute Error) 계산
    mae = torch.abs(test_preds - y_test_tensor).mean().item()

print(f"테스트 MSE:  {test_loss.item():.4f}")
print(f"테스트 MAE:  {mae:.4f} ($100,000 단위)")
print(f"테스트 MAE:  ${mae * 100000:,.0f} (실제 금액)")

테스트 MSE:  0.3018
테스트 MAE:  0.3829 ($100,000 단위)
테스트 MAE:  $38,288 (실제 금액)


In [13]:
# [4. 저장] 서빙에 필요한 두 가지를 models/ 에 저장합니다
#   1) 모델 가중치(state_dict) 2) 전처리 파라미터(학습 데이터의 평균/표준편차, 피처 순서)
#   서버에서도 학습 때와 똑같이 정규화해야 같은 예측이 나옵니다
import os
os.makedirs("models", exist_ok=True)

# 모델 가중치 저장
torch.save(model.state_dict(), "models/housing_model.pth")  # *your code* — state_dict 저장
print(f"✅ 모델 저장: models/housing_model.pth ({os.path.getsize('models/housing_model.pth')/1024:.1f} KB)")

# 전처리 파라미터 저장 (배포 시 필수!)
preprocessing_params = {
    "mean": train_mean.tolist(),
    "std": train_std.tolist(),
    "feature_names": feature_names,
}

import json
with open("models/housing_preprocessing.json", "w") as f:
    json.dump(preprocessing_params, f, indent=2)

print(f"✅ 전처리 파라미터 저장: models/housing_preprocessing.json")

✅ 모델 저장: models/housing_model.pth (13.3 KB)
✅ 전처리 파라미터 저장: models/housing_preprocessing.json


In [14]:
%%writefile app/housing_model.py
# [5. 모듈화 app/housing_model.py] 서버가 import 할 모델 파일
#   - HousingModel     : 학습 때와 똑같은 구조여야 state_dict 를 올릴 수 있습니다
#   - HousingPredictor : 모델 로드 + 전처리 + 추론을 한 클래스로 묶습니다 (서버는 predict() 만 부름)
"""
Day 5 - 주택 가격 예측 모델 정의 + 추론 함수
"""
import json
import torch
import torch.nn as nn
import numpy as np


class HousingModel(nn.Module):
    """캘리포니아 주택 가격 예측 모델"""
    def __init__(self, input_dim=8):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x)


class HousingPredictor:
    """모델 로드 + 전처리 + 추론을 캡슐화한 클래스"""

    def __init__(self, model_path: str, preprocessing_path: str):
        # 전처리 파라미터 로드
        with open(preprocessing_path, "r") as f:
            params = json.load(f)
        self.mean = np.array(params["mean"])
        self.std = np.array(params["std"])
        self.feature_names = params["feature_names"]

        # 모델 로드
        self.model = HousingModel(input_dim=len(self.feature_names))
        self.model.load_state_dict(
            # map_location="cpu" : GPU 에서 저장한 가중치도 CPU 서버에서 읽게 함
            # weights_only=True : 가중치만 읽어 임의 코드 실행을 막음 (보안)
            torch.load(model_path, map_location="cpu", weights_only=True)
        )
        self.model.eval()

    # 추론 순서 : dict -> 학습 때 피처 순서로 배열 -> 학습 통계로 정규화 -> (1, 8) 텐서 -> 모델
    def predict(self, features: dict) -> dict:
        """
        피처 딕셔너리를 받아 가격을 예측합니다.

        Args:
            features: {"MedInc": 3.5, "HouseAge": 25, ...}
        Returns:
            {"predicted_price": 2.35, "predicted_price_usd": 235000}
        """
        # 피처를 올바른 순서로 배열
        values = [features[name] for name in self.feature_names]  # *your code* — 피처 순서 맞추기

        # 정규화
        values = np.array(values, dtype=np.float32)
        normalized = (values - self.mean) / self.std              # *your code* — 정규화 적용

        # 추론
        input_tensor = torch.FloatTensor(normalized).unsqueeze(0)  # (1, 8)
        with torch.no_grad():
            output = self.model(input_tensor)

        price = output.item()
        price = max(price, 0.0)  # 음수 방지

        return {
            "predicted_price": round(price, 4),
            "predicted_price_usd": int(price * 100000),
        }

Overwriting app/housing_model.py


In [15]:
# [5.1 모듈 테스트] 서버에 올리기 전에 파일로 만든 모듈을 직접 불러 예측해 봅니다
#   테스트 데이터 첫 샘플로 예측값과 실제값을 비교합니다
# 모듈 테스트
import sys
sys.path.insert(0, ".")

from app.housing_model import HousingPredictor

predictor = HousingPredictor(
    model_path="models/housing_model.pth",
    preprocessing_path="models/housing_preprocessing.json",
)

# 테스트 데이터의 첫 번째 샘플로 테스트
sample_features = {name: float(X_test[0, i]) for i, name in enumerate(feature_names)}
print(f"입력 피처: {sample_features}")

result = predictor.predict(sample_features)
print(f"예측 가격: ${result['predicted_price_usd']:,}")
print(f"실제 가격: ${int(y_test[0] * 100000):,}")

입력 피처: {'MedInc': 3.7917, 'HouseAge': 40.0, 'AveRooms': 4.959798994974874, 'AveBedrms': 1.0301507537688441, 'Population': 1039.0, 'AveOccup': 2.6105527638190953, 'Latitude': 38.24, 'Longitude': -122.64}
예측 가격: $206,687
실제 가격: $151,600


In [16]:
%%writefile app/housing_schemas.py
# [6. 요청/응답 규격 app/housing_schemas.py]
#   Field(gt, ge, le) 로 값의 범위를 제한합니다. 범위를 벗어나면 FastAPI 가 자동으로 422 를 돌려줍니다
#   json_schema_extra 의 예시는 Swagger UI(/docs) 의 기본 요청값으로 보입니다
"""
Day 5 - 주택 가격 예측 API 스키마
"""
from pydantic import BaseModel, Field


class HousingRequest(BaseModel):
    """주택 가격 예측 요청"""
    MedInc: float = Field(..., gt=0, description="중위 소득")                        # *your code* — gt=0 설정
    HouseAge: float = Field(..., ge=0, le=100, description="주택 연식 (년)")          # *your code* — ge, le 범위
    AveRooms: float = Field(..., gt=0, description="평균 방 수")
    AveBedrms: float = Field(..., gt=0, description="평균 침실 수")
    Population: float = Field(..., gt=0, description="인구")
    AveOccup: float = Field(..., gt=0, description="평균 거주 인원")
    Latitude: float = Field(..., ge=32, le=42, description="위도 (캘리포니아 범위)")   # *your code* — 캘리포니아 위도 범위
    Longitude: float = Field(..., ge=-125, le=-114, description="경도 (캘리포니아 범위)")

    model_config = {
        "json_schema_extra": {
            "examples": [
                {
                    "MedInc": 3.5,
                    "HouseAge": 25.0,
                    "AveRooms": 5.0,
                    "AveBedrms": 1.0,
                    "Population": 1500.0,
                    "AveOccup": 3.0,
                    "Latitude": 37.5,
                    "Longitude": -122.0,
                }
            ]
        }
    }


class HousingResponse(BaseModel):
    """주택 가격 예측 응답"""
    success: bool = Field(description="요청 처리 성공 여부")
    predicted_price: float = Field(description="예측 가격 ($100,000 단위)")
    predicted_price_usd: int = Field(description="예측 가격 (USD)")
    input_features: dict = Field(description="입력된 피처 값")

Overwriting app/housing_schemas.py


In [17]:
%%writefile app/housing_api.py
# [7. API 서버 app/housing_api.py]
#   - startup 에서 모델을 한 번만 올립니다
#   - 추론은 run_in_executor 로 별도 스레드에서 돌려 이벤트 루프가 멈추지 않게 합니다
#   - GET /health : 모델 로드 여부 / POST /predict : 주택 정보 -> 예측 가격
"""
Day 5 - 주택 가격 예측 FastAPI 서버
"""
import asyncio
from concurrent.futures import ThreadPoolExecutor

from fastapi import FastAPI, HTTPException

from app.housing_schemas import HousingRequest, HousingResponse
from app.housing_model import HousingPredictor
from app.logger_config import setup_logger
from app.error_handlers import register_error_handlers
from app.middleware import RequestLoggingMiddleware


# ===== 설정 =====
logger = setup_logger("housing_api")

app = FastAPI(
    title="California Housing Price API",
    description="캘리포니아 주택 가격을 예측하는 API",
    version="1.0.0",
)

app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)

# 추론 전용 스레드풀 (Day 3에서 배운 패턴)
inference_executor = ThreadPoolExecutor(max_workers=4, thread_name_prefix="housing")  # *your code* — 스레드풀 생성

# ===== 모델 로드 =====
MODEL_PATH = "models/housing_model.pth"
PREPROCESS_PATH = "models/housing_preprocessing.json"
predictor = None


# 서버가 뜰 때 한 번만 실행 -> 요청마다 모델을 읽지 않습니다
@app.on_event("startup")
async def startup():
    global predictor
    logger.info("주택 가격 모델 로드 중...")
    predictor = HousingPredictor(MODEL_PATH, PREPROCESS_PATH)  # *your code* — HousingPredictor 인스턴스 생성
    logger.info("모델 로드 완료")


# ===== 엔드포인트 =====

@app.get("/health", tags=["System"])
async def health_check():
    return {
        "status": "healthy" if predictor is not None else "loading",
        "model": "California Housing",
    }


@app.post("/predict", response_model=HousingResponse, tags=["Prediction"])
async def predict_housing(request: HousingRequest):
    """주택 정보를 받아 가격을 예측합니다."""
    if predictor is None:
        # 503 : 서버는 살아 있지만 아직 요청을 처리할 준비가 안 됨
        raise HTTPException(status_code=503, detail="모델이 아직 로드되지 않았습니다.")

    # 요청 데이터를 딕셔너리로 변환
    features = request.model_dump()  # *your code* — Pydantic 모델 → dict

    try:
        # 추론 (별도 스레드에서 실행 — Day 3 패턴)
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(       # *your code* — run_in_executor 사용
            inference_executor,
            predictor.predict,
            features,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"추론 실패: {str(e)}")

    return HousingResponse(
        success=True,
        predicted_price=result["predicted_price"],
        predicted_price_usd=result["predicted_price_usd"],
        input_features=features,
    )

Overwriting app/housing_api.py


In [18]:
# [8. 실행 환경 확인] 지금 커널의 파이썬 경로와 버전 (.venv-wsl312 이어야 합니다)
import sys
print(sys.executable)
print(sys.version)

/mnt/c/EPA1st/model-serving-course/.venv-wsl312/bin/python
3.12.16 (main, Oct 26 2026, 14:08:45) [Clang 23.1.3 ]


In [19]:
# [8. 실행 환경 확인] 서버 실행에 필요한 패키지(nest_asyncio, uvicorn, fastapi)가 import 되는지 확인
import nest_asyncio
import uvicorn
import fastapi

print("Python:", __import__("sys").executable)
print("nest_asyncio:", nest_asyncio.__file__)
print("uvicorn:", uvicorn.__version__)
print("fastapi:", fastapi.__version__)

Python: /mnt/c/EPA1st/model-serving-course/.venv-wsl312/bin/python
nest_asyncio: /mnt/c/EPA1st/model-serving-course/.venv-wsl312/lib/python3.12/site-packages/nest_asyncio.py
uvicorn: 0.53.0
fastapi: 0.141.1


In [20]:
# [8. 실행 환경 확인] nest_asyncio : 주피터가 이미 돌리는 이벤트 루프 안에서 uvicorn 을 돌릴 수 있게 합니다
import nest_asyncio
nest_asyncio.apply()

print("Jupyter 비동기 이벤트 루프 설정 완료")

Jupyter 비동기 이벤트 루프 설정 완료


In [21]:
# [8. 실행 환경 확인] 패키지가 없을 때만 필요한 설치 셀 (이미 깔려 있으면 아무 일도 하지 않음)
%pip install nest_asyncio
%pip install uvicorn

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [22]:
# [8. 실행 환경 확인] 설치 후 커널 파이썬 경로 재확인
import sys
print(sys.executable)

/mnt/c/EPA1st/model-serving-course/.venv-wsl312/bin/python


In [23]:
# [8. 실행 환경 확인] 설치된 버전 확인 (import 없이 메타데이터로 확인)
import importlib.metadata

print("nest_asyncio:", importlib.metadata.version("nest-asyncio"))
print("uvicorn:", importlib.metadata.version("uvicorn"))

nest_asyncio: 1.6.0
uvicorn: 0.53.0


In [24]:
# [9. 서버 띄우기] 노트북 안에서 uvicorn 을 백그라운드 스레드로 실행합니다 (8000번 포트)
#   daemon=True 라 커널을 끄면 서버도 함께 꺼집니다
#   8000번을 다른 서버가 쓰고 있으면 기동이 실패하니 먼저 확인합니다
# ⚠️ 이전 서버가 실행 중이면 커널을 재시작하세요.

import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_server():
    uvicorn.run("app.housing_api:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("✅ 서버 시작됨: http://localhost:8000")
print("📄 Swagger UI: http://localhost:8000/docs")

INFO:     Started server process [235719]


INFO:     Waiting for application startup.


2026-11-04 15:08:06 INFO     [housing_api] 주택 가격 모델 로드 중...


2026-11-04 15:08:06 INFO     [housing_api] 모델 로드 완료


INFO:     Application startup complete.


INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ 서버 시작됨: http://localhost:8000
📄 Swagger UI: http://localhost:8000/docs


In [25]:
# [9.1 API 테스트] 헬스체크 : 모델이 올라왔으면 status 가 healthy
import requests, json

# 헬스체크
resp = requests.get("http://localhost:8000/health")
print(f"헬스체크: {resp.json()}")

INFO:     127.0.0.1:48540 - "GET /health HTTP/1.1" 200 OK


헬스체크: {'status': 'healthy', 'model': 'California Housing'}


In [26]:
# [9.1 API 테스트] 정상 요청 하나를 보내 예측 가격과 전체 응답을 확인합니다
# 추론 테스트
sample_request = {
    "MedInc": 3.5,
    "HouseAge": 25.0,
    "AveRooms": 5.0,
    "AveBedrms": 1.0,
    "Population": 1500.0,
    "AveOccup": 3.0,
    "Latitude": 37.5,
    "Longitude": -122.0,
}

resp = requests.post("http://localhost:8000/predict", json=sample_request)
result = resp.json()

print(f"상태 코드: {resp.status_code}")
print(f"예측 가격: ${result['predicted_price_usd']:,}")
print(f"전체 응답:")
print(json.dumps(result, indent=2, ensure_ascii=False))

INFO:     127.0.0.1:48550 - "POST /predict HTTP/1.1" 200 OK


상태 코드: 200
예측 가격: $180,558
전체 응답:
{
  "success": true,
  "predicted_price": 1.8056,
  "predicted_price_usd": 180558,
  "input_features": {
    "MedInc": 3.5,
    "HouseAge": 25.0,
    "AveRooms": 5.0,
    "AveBedrms": 1.0,
    "Population": 1500.0,
    "AveOccup": 3.0,
    "Latitude": 37.5,
    "Longitude": -122.0
  }
}


In [27]:
# [9.1 API 테스트] 잘못된 요청은 서버가 죽지 않고 422 를 돌려줘야 합니다
# 에러 테스트: 필수 필드 누락
resp = requests.post("http://localhost:8000/predict", json={"MedInc": 3.5})
print(f"필드 누락 → 상태: {resp.status_code}")

# 에러 테스트: 범위 초과
resp = requests.post("http://localhost:8000/predict", json={
    **sample_request, "Latitude": 50.0  # 캘리포니아 범위 초과
})
print(f"범위 초과 → 상태: {resp.status_code}")

INFO:     127.0.0.1:48552 - "POST /predict HTTP/1.1" 422 Unprocessable Entity


POST /predict -> 422 (머리까지 0.00초, 끝까지 0.00초, 완료)


필드 누락 → 상태: 422
INFO:     127.0.0.1:48562 - "POST /predict HTTP/1.1" 422 Unprocessable Entity


POST /predict -> 422 (머리까지 0.00초, 끝까지 0.00초, 완료)


범위 초과 → 상태: 422


In [28]:
%%writefile frontend/app_housing.py
# [10. 화면 frontend/app_housing.py] Streamlit 대시보드
#   - 사이드바 : /health 로 서버 연결 상태 표시
#   - 왼쪽 : 피처 8개 입력 / 오른쪽 : 예측 버튼과 결과
#   - 결과를 st.session_state 에 저장해 화면이 다시 실행돼도 사라지지 않게 합니다
"""
Day 5 - 캘리포니아 주택 가격 예측 대시보드
"""
import streamlit as st
import requests

# ===== 페이지 설정 =====
st.set_page_config(
    page_title="주택 가격 예측",
    page_icon="🏠",
    layout="wide",
)

# ===== API 호출 함수 (Day 4 패턴) =====
API_BASE = "http://localhost:8000"

def call_api(url, json_data=None, method="post"):
    try:
        if method == "get":
            resp = requests.get(url, timeout=10)
        else:
            resp = requests.post(url, json=json_data, timeout=30)
        resp.raise_for_status()                      # *your code* — HTTP 에러 시 예외 발생
        return resp.json()
    except requests.exceptions.ConnectionError:
        st.error("🔌 **서버에 연결할 수 없습니다.** FastAPI 서버를 실행하세요.")
        return None
    except requests.exceptions.HTTPError as e:
        st.error(f"❌ **서버 에러** (HTTP {e.response.status_code})")
        return None
    except Exception as e:
        st.error(f"❌ **오류:** {type(e).__name__}")
        return None


# ===== 사이드바 =====
with st.sidebar:
    st.header("⚙️ 설정")

    health = call_api(f"{API_BASE}/health", method="get")
    if health and health.get("status") == "healthy":
        st.success("🟢 서버 연결됨")
        server_ok = True
    else:
        st.error("🔴 서버 연결 실패")
        server_ok = False

    st.divider()
    st.caption("California Housing Price Predictor")
    st.caption("Day 5 — 프로젝트 1")


# ===== 메인 영역 =====
st.title("🏠 캘리포니아 주택 가격 예측")
st.write("주택 정보를 입력하면 예상 가격을 예측합니다.")

col_input, col_result = st.columns(2)

# ----- 입력 영역 -----
with col_input:
    st.subheader("📋 주택 정보 입력")

    # 소득 & 주택 연식
    c1, c2 = st.columns(2)
    with c1:
        med_inc = st.number_input(
            "중위 소득 (MedInc)",
            min_value=0.1, max_value=20.0, value=3.5, step=0.1,   # *your code* — 범위와 기본값
        )
    with c2:
        house_age = st.number_input(
            "주택 연식 (HouseAge)",
            min_value=0.0, max_value=100.0, value=25.0, step=1.0,
        )

    # 방 수 & 침실 수
    c1, c2 = st.columns(2)
    with c1:
        ave_rooms = st.number_input(
            "평균 방 수 (AveRooms)",
            min_value=0.1, max_value=50.0, value=5.0, step=0.1,
        )
    with c2:
        ave_bedrms = st.number_input(
            "평균 침실 수 (AveBedrms)",
            min_value=0.1, max_value=20.0, value=1.0, step=0.1,
        )

    # 인구 & 거주 인원
    c1, c2 = st.columns(2)
    with c1:
        population = st.number_input(
            "인구 (Population)",
            min_value=1.0, max_value=50000.0, value=1500.0, step=100.0,
        )
    with c2:
        ave_occup = st.number_input(
            "평균 거주 인원 (AveOccup)",
            min_value=0.1, max_value=20.0, value=3.0, step=0.1,
        )

    # 위치
    c1, c2 = st.columns(2)
    with c1:
        latitude = st.number_input(
            "위도 (Latitude)",
            min_value=32.0, max_value=42.0, value=37.5, step=0.1,   # *your code* — 캘리포니아 범위
        )
    with c2:
        longitude = st.number_input(
            "경도 (Longitude)",
            min_value=-125.0, max_value=-114.0, value=-122.0, step=0.1,
        )


# ----- 결과 영역 -----
with col_result:
    st.subheader("📊 예측 결과")

    if not server_ok:
        st.error("서버에 연결할 수 없습니다.")
    else:
        if st.button("🚀 가격 예측", type="primary", use_container_width=True):
            # 요청 데이터 구성
            request_data = {                            # *your code* — 입력값을 dict로 구성
                "MedInc": med_inc,
                "HouseAge": house_age,
                "AveRooms": ave_rooms,
                "AveBedrms": ave_bedrms,
                "Population": population,
                "AveOccup": ave_occup,
                "Latitude": latitude,
                "Longitude": longitude,
            }

            with st.spinner("예측 중..."):
                result = call_api(f"{API_BASE}/predict", json_data=request_data)

            if result:
                st.session_state["last_housing_result"] = result

        # 결과 표시
        if "last_housing_result" in st.session_state:
            result = st.session_state["last_housing_result"]

            # 가격 메트릭
            st.metric(
                label="예상 주택 가격",
                value=f"${result['predicted_price_usd']:,}",
            )

            st.caption(f"모델 출력값: {result['predicted_price']} ($100,000 단위)")

            # 입력 피처 확인
            with st.expander("📋 입력된 피처 확인"):
                for key, value in result["input_features"].items():
                    st.write(f"**{key}**: {value}")

Overwriting frontend/app_housing.py


In [1]:
# [10.1 실행 방법] 이 셀은 '터미널에서 실행할 명령' 예시입니다
#   !uvicorn 은 서버가 끝나지 않아 셀이 멈추고, 위에서 띄운 서버와 포트(8000)가 겹칩니다
#   노트북에서는 실행하지 말고 각 명령을 터미널 두 개에 나눠 실행합니다
# 터미널 1: 백엔드
!uvicorn app.housing_api:app --port 8000

# 터미널 2: 프론트엔드
!streamlit run frontend/app_housing.py --server.port 8501

INFO:     Started server process [17020]
INFO:     Waiting for application startup.
2026-11-04 00:08:09 INFO     [housing_api] 주택 가격 모델 로드 중...
2026-11-04 00:08:09 INFO     [housing_api] 모델 로드 완료
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
/bin/bash: line 1: streamlit: command not found


In [29]:
# [10.1 노트북에서 화면 띄우기] streamlit 을 별도 프로세스로 백그라운드 실행합니다 (8501번 포트)
#   커널을 꺼도 이 프로세스는 남으니, 다 쓰면 proc.terminate() 로 내립니다
# 또는 노트북에서 백그라운드 실행
import subprocess, time

proc = subprocess.Popen(
    ["streamlit", "run", "frontend/app_housing.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(3)
print("✅ 프론트엔드: http://localhost:8501")

✅ 프론트엔드: http://localhost:8501


In [30]:
# ==========================================================================
# [11. 통합 테스트] 서버가 떠 있는 상태에서 정상/에러/동시/헬스체크를 차례로 확인합니다
# ==========================================================================
# ⚠️ FastAPI 서버가 실행 중이어야 합니다.

import requests
import json
import time

API_BASE = "http://localhost:8000"

print("=" * 60)
print("  통합 테스트")
print("=" * 60)

  통합 테스트


In [31]:
# [테스트 1] 입력이 달라지면 예측도 합리적으로 달라지는지 (저소득 < 평균 < 고소득)
# 다양한 입력으로 모델 동작 확인
test_cases = [
    {"name": "저소득 지역", "MedInc": 1.5, "HouseAge": 40, "AveRooms": 4.0, "AveBedrms": 1.0,
     "Population": 2000, "AveOccup": 3.5, "Latitude": 34.0, "Longitude": -118.0},
    {"name": "고소득 지역", "MedInc": 10.0, "HouseAge": 10, "AveRooms": 8.0, "AveBedrms": 2.0,
     "Population": 500, "AveOccup": 2.0, "Latitude": 37.8, "Longitude": -122.4},
    {"name": "평균적 주택", "MedInc": 3.5, "HouseAge": 25, "AveRooms": 5.0, "AveBedrms": 1.0,
     "Population": 1500, "AveOccup": 3.0, "Latitude": 37.5, "Longitude": -122.0},
]

print("\n[테스트 1] 정상 요청 — 다양한 입력")
print(f"{'케이스':<15} {'예측 가격':>12}")
print("-" * 30)

for case in test_cases:
    name = case.pop("name")
    resp = requests.post(f"{API_BASE}/predict", json=case)    # *your code* — POST 요청
    result = resp.json()
    print(f"{name:<15} ${result['predicted_price_usd']:>10,}")
    case["name"] = name  # 복원


[테스트 1] 정상 요청 — 다양한 입력
케이스                    예측 가격
------------------------------
INFO:     127.0.0.1:36056 - "POST /predict HTTP/1.1" 200 OK


저소득 지역          $   110,289
INFO:     127.0.0.1:36070 - "POST /predict HTTP/1.1" 200 OK


고소득 지역          $   381,807
INFO:     127.0.0.1:36082 - "POST /predict HTTP/1.1" 200 OK


평균적 주택          $   180,558


In [32]:
# [테스트 2] 에러 상황 : 필드 누락/범위 초과/음수 -> 422, JSON 이 아닌 본문 -> 422
print("\n[테스트 2] 에러 상황")

# 필수 필드 누락
resp = requests.post(f"{API_BASE}/predict", json={"MedInc": 3.5})
print(f"  필드 누락      → HTTP {resp.status_code}")

# 범위 초과 (위도)
bad_request = {
    "MedInc": 3.5, "HouseAge": 25, "AveRooms": 5, "AveBedrms": 1,
    "Population": 1500, "AveOccup": 3, "Latitude": 50.0, "Longitude": -122.0,  # 위도 초과
}
resp = requests.post(f"{API_BASE}/predict", json=bad_request)
print(f"  위도 범위 초과  → HTTP {resp.status_code}")

# 음수 값
bad_request2 = {
    "MedInc": -1.0, "HouseAge": 25, "AveRooms": 5, "AveBedrms": 1,
    "Population": 1500, "AveOccup": 3, "Latitude": 37.5, "Longitude": -122.0,
}
resp = requests.post(f"{API_BASE}/predict", json=bad_request2)
print(f"  소득 음수      → HTTP {resp.status_code}")

# JSON이 아닌 요청
resp = requests.post(f"{API_BASE}/predict", data="not json")
print(f"  잘못된 포맷    → HTTP {resp.status_code}")


[테스트 2] 에러 상황
INFO:     127.0.0.1:36096 - "POST /predict HTTP/1.1" 422 Unprocessable Entity


POST /predict -> 422 (머리까지 0.00초, 끝까지 0.00초, 완료)


  필드 누락      → HTTP 422
INFO:     127.0.0.1:36112 - "POST /predict HTTP/1.1" 422 Unprocessable Entity


POST /predict -> 422 (머리까지 0.00초, 끝까지 0.00초, 완료)


  위도 범위 초과  → HTTP 422
INFO:     127.0.0.1:36122 - "POST /predict HTTP/1.1" 422 Unprocessable Entity


POST /predict -> 422 (머리까지 0.00초, 끝까지 0.00초, 완료)


  소득 음수      → HTTP 422
INFO:     127.0.0.1:36138 - "POST /predict HTTP/1.1" 422 Unprocessable Entity


POST /predict -> 422 (머리까지 0.00초, 끝까지 0.00초, 완료)


  잘못된 포맷    → HTTP 422


In [33]:
# [테스트 3] 동시 요청 8개 : 추론 스레드풀(max_workers=4)이 나눠 처리하고 모두 200 이어야 합니다
from concurrent.futures import ThreadPoolExecutor, as_completed

def send_predict(i):
    case = test_cases[i % len(test_cases)].copy()
    case.pop("name", None)
    start = time.time()
    resp = requests.post(f"{API_BASE}/predict", json=case, timeout=30)
    return {"id": i+1, "elapsed": round(time.time() - start, 3), "status": resp.status_code}

print("\n[테스트 3] 동시 요청 (8개)")
start = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = [ex.submit(send_predict, i) for i in range(8)]
    results = [f.result() for f in as_completed(futures)]

total = round(time.time() - start, 2)
for r in sorted(results, key=lambda x: x["id"]):
    print(f"  요청 #{r['id']}: {r['elapsed']}초 (HTTP {r['status']})")
print(f"  전체: {total}초")


[테스트 3] 동시 요청 (8개)
INFO:     127.0.0.1:36170 - "POST /predict HTTP/1.1" 200 OK


INFO:     127.0.0.1:36174 - "POST /predict HTTP/1.1" 200 OK


INFO:     127.0.0.1:36188 - "POST /predict HTTP/1.1" 200 OK


INFO:     127.0.0.1:36152 - "POST /predict HTTP/1.1" 200 OK


INFO:     127.0.0.1:36226 - "POST /predict HTTP/1.1" 200 OK


INFO:     127.0.0.1:36204 - "POST /predict HTTP/1.1" 200 OK


INFO:     127.0.0.1:36158 - "POST /predict HTTP/1.1" 200 OK


INFO:     127.0.0.1:36220 - "POST /predict HTTP/1.1" 200 OK


  요청 #1: 0.025초 (HTTP 200)
  요청 #2: 0.021초 (HTTP 200)
  요청 #3: 0.019초 (HTTP 200)
  요청 #4: 0.016초 (HTTP 200)
  요청 #5: 0.016초 (HTTP 200)
  요청 #6: 0.019초 (HTTP 200)
  요청 #7: 0.021초 (HTTP 200)
  요청 #8: 0.016초 (HTTP 200)
  전체: 0.03초


In [34]:
# [테스트 4] 테스트를 다 돌린 뒤에도 서버가 살아 있는지 헬스체크
print("\n[테스트 4] 헬스체크")
resp = requests.get(f"{API_BASE}/health")
print(f"  상태: {resp.json()}")


[테스트 4] 헬스체크
INFO:     127.0.0.1:36236 - "GET /health HTTP/1.1" 200 OK


  상태: {'status': 'healthy', 'model': 'California Housing'}


In [35]:
# [테스트 결과 종합]
print("\n" + "=" * 60)
print("  테스트 결과 종합")
print("=" * 60)
print("  ✅ 정상 요청: 다양한 입력에서 합리적인 가격 반환")
print("  ✅ 에러 처리: 잘못된 입력에 422/400 반환, 서버 안 죽음")
print("  ✅ 동시 처리: 8개 동시 요청 정상 처리")
print("  ✅ 헬스체크: 서버 상태 정상")


  테스트 결과 종합
  ✅ 정상 요청: 다양한 입력에서 합리적인 가격 반환
  ✅ 에러 처리: 잘못된 입력에 422/400 반환, 서버 안 죽음
  ✅ 동시 처리: 8개 동시 요청 정상 처리
  ✅ 헬스체크: 서버 상태 정상
